# Average standardized income per municipality

This notebook prepares the CBS average standardized income data for Dutch municipalities (2024), creates a map and saves the processed data for further analysis.

**Input (`data/raw/`):**
- `inkomens_per_gemeente.csv`: CBS standardized income per municipality (2024)
- `gemeenten_2024.gpkg`: municipal boundaries for the map

**Output:**
- `data/processed/income_per_municipality.csv`: municipality and average standardized income in euros
- `figures/map_income_per_municipality.png`: income map

*Note: this notebook uses the same income measure as the existing charging points/households/income notebook. It does not change that notebook's output.*

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("..") if Path("../data/raw").exists() else Path(".")


## 1. Load the income data

In [ ]:
income = pd.read_csv(root / "data/raw/inkomens_per_gemeente.csv", sep=";", encoding="utf-8-sig")

income = income[[
    "Regio's",
    "Inkomen/Gemiddeld gestandaardiseerd inkomen (1 000 euro)"
]].copy()

income.columns = ["municipality", "average_standardized_income_eur"]

# CBS records income in thousands of euros; convert to euros
income["average_standardized_income_eur"] = pd.to_numeric(
    income["average_standardized_income_eur"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
) * 1000

income["municipality"] = income["municipality"].astype(str).str.strip()
income = income.dropna(subset=["average_standardized_income_eur"])
print("Rows with income data:", len(income))
display(income.head())

## 2. Harmonise municipality names

In [ ]:
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "'s-Gravenhage (gemeente)": "Den Haag",
    "Groningen (gemeente)": "Groningen",
    "Utrecht (gemeente)": "Utrecht",
    "Beek (L.)": "Beek",
    "Stein (L.)": "Stein",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk"
}

income["municipality"] = income["municipality"].replace(name_mapping)

# Keep municipality-level observations
income = income[income["municipality"].ne("Totaal")]
print("Duplicate municipality names:", income["municipality"].duplicated().sum())
print("Municipalities:", income["municipality"].nunique())


## 3. Load municipal boundaries and join the data

In [ ]:
municipalities = gpd.read_file(root / "data/raw/gemeenten_2024.gpkg", layer="gemeente_gegeneraliseerd")

map_name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein"
}
municipalities["municipality"] = municipalities["statnaam"].replace(map_name_mapping)
map_gdf = municipalities.merge(income, on="municipality", how="left")

print("Municipalities without income data:", map_gdf["average_standardized_income_eur"].isna().sum())
print("Income municipalities not on the map:", set(income["municipality"]) - set(municipalities["municipality"]))

## 4. Create the map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))
map_gdf.plot(
    column="average_standardized_income_eur",
    cmap="Reds",
    vmin=34000,
    vmax=62000,
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Average standardized income (€)", "shrink": 0.65},
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax
)
ax.set_title("Average standardized income per municipality (2024)", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
(root / "figures").mkdir(exist_ok=True)
plt.savefig(root / "figures/map_income_per_municipality.png", dpi=200)
plt.show()

## 5. Save for the analysis

In [ ]:
# Keep the output filename stable for other notebooks
output = income[["municipality", "average_standardized_income_eur"]].copy()

# Stop if names are not unique
assert not output["municipality"].duplicated().any(), "Duplicate municipality names found"

(root / "data/processed").mkdir(parents=True, exist_ok=True)
output.to_csv(root / "data/processed/income_per_municipality.csv", index=False)
print("Saved", len(output), "municipalities")
display(output.head())